In [3]:
import ee
import geemap
import geopandas as gpd
import pandas as pd

try:
	ee.Initialize()
except Exception:
	ee.Authenticate()
	ee.Initialize()


Successfully saved authorization token.


In [ ]:
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"
gdf = gpd.read_file(UC_SHP).to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()
YEAR, MONTH = 2024, 10

In [ ]:
# ---------- AQI (PM2.5) from CAMS ----------
# Source: ECMWF/CAMS/NRT  (band: particulate_matter_d_less_than_25_um_surface, kg/m^3)
COL_AQI = "ECMWF/CAMS/NRT"

id_field = "UC" if "UC" in gdf.columns else ("uc_id" if "uc_id" in gdf.columns else gdf.columns[0])
gdf_fc = geemap.gdf_to_ee(gdf[[id_field, "geometry"]])

# PM2.5 -> AQI piecewise (ee.Image in, ee.Image out)
def pm25_to_aqi(pm_img):
    breaks = [
        (0.0, 12.0,   0,   50),
        (12.1, 35.4,  51,  100),
        (35.5, 55.4,  101, 150),
        (55.5, 150.4, 151, 200),
        (150.5, 250.4,201, 300),
        (250.5, 500.4,301, 500)
    ]
    def seg(p):
        cl, ch, il, ih = p
        return (pm_img.subtract(cl)
                    .multiply((ih - il) / (ch - cl))
                    .add(il)
                ).updateMask(pm_img.gte(cl).And(pm_img.lte(ch)))
    return ee.ImageCollection([seg(b) for b in breaks]).max().rename("AQI")

# Build CAMS collection for month
d0 = ee.Date.fromYMD(YEAR, MONTH, 1)
d1 = d0.advance(1, "month")
cams = (ee.ImageCollection(COL_AQI)
        .filterBounds(region)
        .filterDate(d0, d1))

# Select PM2.5 band and take model hour 0 per image, then convert kg/m^3 -> µg/m^3
pm_band = "particulate_matter_d_less_than_25_um_surface"

def to_pm_ug(img):
    img0 = img.select(pm_band)
    pm_ug = img0.multiply(1e9).rename("PM25_ugm3")
    return pm_ug.copyProperties(img, img.propertyNames())

# Some images may not have the band; filter them out safely
cams_with_pm = cams.filter(ee.Filter.listContains("system:band_names", pm_band))
daily_pm = cams_with_pm.map(to_pm_ug)

# Map to AQI per image
daily_aqi = daily_pm.map(lambda im: pm25_to_aqi(im).copyProperties(im, im.propertyNames()))

# ---- UC-level monthly aggregates ----
# Reduce each day over UCs, then summarize across days

def reduce_per_day(imgcol, bandname):
    per_day = imgcol.map(lambda im:
        im.reduceRegions(collection=gdf_fc, reducer=ee.Reducer.mean(), scale=5000)
          .map(lambda f: f.set({"date": im.date().format("YYYY-MM-dd"), id_field: f.get(id_field)}))
    )
    return ee.FeatureCollection(per_day.flatten())

pm_per_day  = reduce_per_day(daily_pm,  "PM25_ugm3")
aqi_per_day = reduce_per_day(daily_aqi, "AQI")

# Group across days per UC

def group_stats(per_day_fc, base_name):
    reducer = (ee.Reducer.mean()
               .combine(ee.Reducer.median(), '', True)
               .combine(ee.Reducer.max(), '', True)
               .group(groupField=1))
    groups = per_day_fc.reduceColumns(
        selectors=["mean", id_field],
        reducer=reducer
    ).get("groups")
    groups = ee.List(ee.Algorithms.If(groups, groups, ee.List([])))
    def to_feature(g):
        gdict = ee.Dictionary(g)
        props = ee.Dictionary({ id_field: gdict.get("group") })
        props = ee.Dictionary(ee.Algorithms.If(gdict.contains("mean"), props.set(f"{base_name}_mean", gdict.get("mean")), props))
        props = ee.Dictionary(ee.Algorithms.If(gdict.contains("median"), props.set(f"{base_name}_median", gdict.get("median")), props))
        props = ee.Dictionary(ee.Algorithms.If(gdict.contains("max"), props.set(f"{base_name}_max", gdict.get("max")), props))
        return ee.Feature(None, props)
    return ee.FeatureCollection(groups.map(to_feature))

pm_month  = group_stats(pm_per_day,  "PM25")
aqi_month = group_stats(aqi_per_day, "AQI")

# --- Interpolate/smooth the monthly mean AQI for mapping and per-UC assignment ---
# Create a monthly mean image, fill gaps, and apply a small focal average
monthly_aqi_img = daily_aqi.mean().rename("AQI_mean")
# Fill masked with a constant image of the regional mean value
reg_mean = ee.Number(monthly_aqi_img.reduceRegion(ee.Reducer.mean(), region, 50000, bestEffort=True).get("AQI_mean"))
filled = monthly_aqi_img.unmask(ee.Image.constant(reg_mean))
smoothed = filled.focal_mean(radius=3000, units="meters").rename("AQI_mean_smooth")

# Reduce smoothed AQI over UC polygons (keep geometry for export)
uc_smoothed = smoothed.reduceRegions(
    collection=gdf_fc,
    reducer=ee.Reducer.mean(),
    scale=3000
).map(lambda f: ee.Feature(f.geometry(), {
    id_field: f.get(id_field),
    "AQI_mean_smooth": f.get("mean")
}))

# ---- Join helper (native EE) ----
# Left: features with geometry; Right: properties-only (no geometry required)

def ee_join_on_key(left_fc, right_fc, key):
    left = left_fc.map(lambda f: ee.Feature(f).set('_join_key', ee.Feature(f).get(key)))
    right = ee.FeatureCollection(right_fc).map(lambda f: ee.Feature(None, ee.Feature(f).toDictionary().set('_join_key', ee.Feature(f).get(key))))
    joined = ee.Join.inner().apply(left, right, ee.Filter.equals(leftField='_join_key', rightField='_join_key'))
    def merge(pair):
        pair = ee.Feature(pair)
        primary = ee.Feature(pair.get('primary'))
        secondary = ee.Feature(pair.get('secondary'))
        # combine preserves existing props; remove the temporary key
        combined = primary.toDictionary().combine(secondary.toDictionary(), True).remove(ee.List(['_join_key']))
        return ee.Feature(primary.geometry(), combined)
    return ee.FeatureCollection(joined.map(merge))

# Join tabular stats to smoothed geometry collection
joined = ee_join_on_key(uc_smoothed, aqi_month, id_field)
joined = ee_join_on_key(joined, pm_month,  id_field)

# --- Export interpolated results ---
# 1) GeoJSON with geometry
geojson_path = f"AQI_{YEAR}_{MONTH:02d}_UC_interpolated.geojson"
geemap.ee_export_vector(joined, filename=geojson_path)

# 2) Shapefile (directory with multiple files)
shp_path = f"AQI_{YEAR}_{MONTH:02d}_UC_interpolated.shp"
geemap.ee_export_vector(joined, filename=shp_path)

print("Saved:", geojson_path, "and", shp_path)

Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/AQI/AQI_2024_10_UC_interpolated.geojson
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/AQI/AQI_2024_10_UC_interpolated.geojson
Generating URL ...
Generating URL ...
Please wait ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/AQI/AQI_2024_10_UC_interpolated.shp
Saved: AQI_2024_10_UC_interpolated.geojson and AQI_2024_10_UC_interpolated.shp
Data downloaded to /Users/ahmed/Desktop/Senior Fall 25/SPROJ - Dr Tahir/SPROJ/notebooks/AQI/AQI_2024_10_UC_interpolated.shp
Saved: AQI_2024_10_UC_interpolated.geojson and AQI_2024_10_UC_interpolated.shp


## Interpolated AQI raster for all Lahore (1 km)

- This creates a continuous AQI surface by resampling the monthly CAMS-based AQI to about 1 km using bicubic interpolation, clipped to Lahore.
- Important: This increases visual smoothness but does not add new information beyond the native CAMS resolution (~0.4° ≈ 40–50 km). Treat neighborhood-level variation as interpolated, not measured.


In [ ]:
# Build a continuous AQI surface at ~1 km over Lahore and visualize
import folium
import branca

# 1) Create a filled and smoothed AQI image like before
monthly_aqi_img = daily_aqi.mean().rename('AQI_mean')
reg_mean = ee.Number(monthly_aqi_img.reduceRegion(ee.Reducer.mean(), region, 50000, bestEffort=True).get('AQI_mean'))
filled = monthly_aqi_img.unmask(ee.Image.constant(reg_mean))
# Bicubic resample and slight smoothing to make a continuous surface
resampled = filled.resample('bicubic')
smoothed_1km = resampled.focal_mean(radius=1500, units='meters')

# 2) Define a 1 km scale and visualization parameters
scale_1km = 1000
vis = {
    'min': 0,
    'max': 200,  # adjust if your month has higher values
    'palette': ['#00e400', '#ffff00', '#ff7e00', '#ff0000', '#8f3f97', '#7e0023']
}

# 3) Clip to Lahore and show on a Folium map
aqi_layer = smoothed_1km.clip(region)
img_vis = aqi_layer.visualize(**vis)
map_center = gdf.geometry.unary_union.centroid.y, gdf.geometry.unary_union.centroid.x
m = folium.Map(location=map_center, zoom_start=11, tiles='CartoDB positron')

tile_layer = geemap.ee_tile_layers.EELeafletTileLayer(
    ee_object=aqi_layer,
    vis_params=vis,
    name='AQI (Interpolated ~1km)'
)
m.add_child(tile_layer)

# Add a legend
breaks = [0, 50, 100, 150, 200]
legend = branca.colormap.LinearColormap(colors=vis['palette'][:5], vmin=breaks[0], vmax=breaks[-1])
legend.caption = 'AQI (US EPA) — Interpolated ~1km'
legend.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

/var/folders/80/08nsjy_s5f3brjxw1crm6mbh0000gn/T/ipykernel_37557/1968492410.py:24: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  map_center = gdf.geometry.unary_union.centroid.y, gdf.geometry.unary_union.centroid.x


AttributeError: module 'geemap' has no attribute 'foliumap'

In [18]:
# Optional: Export the interpolated continuous AQI to GeoTIFF (~1 km)
# Note: This makes a raster for the entire Lahore region at ~1 km. Values are AQI index.
export_scale = 1000
export_img = smoothed_1km.clip(region).toFloat().rename('AQI_interpolated_1km')

task = ee.batch.Export.image.toDrive(
    image=export_img,
    description=f"AQI_{YEAR}_{MONTH:02d}_Lahore_1km",
    folder=None,  # default to root of Drive unless you set a folder name
    fileNamePrefix=f"AQI_{YEAR}_{MONTH:02d}_Lahore_1km",
    region=region,
    scale=export_scale,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
task.start()
print('Started export to Google Drive (GeoTIFF ~1 km). Check Tasks tab or Drive.')

EEException: Request had insufficient authentication scopes.